# Challenges of SLMs Reasoning in Math Problems

In [1]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import warnings

warnings.filterwarnings('ignore')

/home/hoang/miniconda3/envs/n-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import os
from pathlib import Path

os.environ["HF_HOME"] = "/mnt/data/hf"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/data/hf/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/mnt/data/hf/hub"
os.environ["TMPDIR"] = "/mnt/data/tmp"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use first GPU only

# Optional: if HF_HOME has no `token` file but `hf auth login` wrote under ~/.cache, expose it for gated Hub downloads
_hf_tok = Path(os.environ["HF_HOME"]) / "token"
_default_tok = Path.home() / ".cache/huggingface/token"
if not _hf_tok.exists() and _default_tok.is_file():
    os.environ["HF_TOKEN"] = _default_tok.read_text().strip()

In [ ]:
torch.random.manual_seed(0)
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

model_id = "microsoft/Phi-4-mini-instruct" #3.8B

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16 if device == "mps" else torch.float32,
    trust_remote_code=True,
).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

messages = [{
    "role": "user",
    "content": "How to solve 3*x^2+4*x+5=1?"
}]   
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

outputs = model.generate(
    **inputs.to(model.device),
    max_new_tokens=32768,
    temperature=0.8,
    top_p=0.95,
    do_sample=True,
)
outputs = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[-1]:])

print(outputs[0])
